# Lab 3: Data Provenance and Auditing

Before I pushed further into modeling, I wanted to document where the data came from and make sure the pipeline was reproducible. If someone else wanted to run my analysis — or if I came back to it in six months — they should be able to verify exactly which files were used and in what state.
This lab isn't about discovering new statistical patterns. It's about making everything I'd built so far trustworthy.

## Scanning the raw files — capturing the file-level metadata

I scanned the five raw CSV files, inferred the year from each filename, and collected file-level metadata: row count, column count, and missing percentage. I stored this in a structured provenance table. The column normalization mapping I built in Lab 1 is the most important documentation in this lab, as the schema differences between years are more significant than they look at first glance.

In [1]:
import hashlib
from pathlib import Path
import re

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = Path("..") / ".." / "data" / "raw"
OUTPUT_DIR = Path("outputs")
PLOTS_DIR = OUTPUT_DIR / "plots"
TABLES_DIR = OUTPUT_DIR / "tables"
PROCESSED_DIR = Path("..") / ".." / "data" / "processed"

PLOTS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

def infer_year_from_filename(name: str) -> int | None:
    match = re.search(r"(19|20)\d{2}", name)
    return int(match.group()) if match else None

files = sorted(DATA_DIR.glob("*.csv"))
if not files:
    raise FileNotFoundError(f"No CSV files found in {DATA_DIR.resolve()}")

rows = []
for fp in files:
    df = pd.read_csv(fp)
    rows.append({
        "file": fp.name,
        "sha256": hashlib.sha256(open(fp, 'rb').read()).hexdigest()[:12],
        "year": infer_year_from_filename(fp.name),
        "rows": df.shape[0],
        "cols": df.shape[1],
        "missing_pct": round(df.isna().mean().mean() * 100, 2),
        "columns": " | ".join(df.columns[:12])
    })

df_meta = pd.DataFrame(rows)
df_meta.sort_values("year")

,file,year,rows,cols,missing_pct,columns
0,2015.csv,2015,158,12,0.00,Country | Region | Happiness Rank | Happiness ...
1,2016.csv,2016,157,13,0.00,Country | Region | Happiness Rank | Happiness ...
2,2017.csv,2017,155,12,0.00,Country | Happiness.Rank | Happiness.Score | W...
3,2018.csv,2018,156,9,0.07,Overall rank | Country or region | Score | GDP...
4,2019.csv,2019,156,9,0.00,Overall rank | Country or region | Score | GDP...


**Observation:**
I found that the row counts are consistent — roughly 155–158 countries per year. The column count varies because the report added and dropped columns across editions. The 2017 file has a small percentage of missing values in the core features that the others don't.
What I noticed is that the schema differences between years are more significant than they look at first glance. It's not just renamed columns — some years include a `Dystopia Residual` column that others don't. Some years break down GDP and social support differently. The normalization mapping in Lab 1 handles this, but documenting it here makes the design decision visible.

## Saving the provenance table — making it reproducible

I need to save this metadata table so later labs can refer to it, proving exactly which files were used.

In [ ]:
df_meta.to_csv(TABLES_DIR / "lab3_provenance.csv", index=False)
df_meta.head()


## Plotting rows by year — a quick visual audit

To make the provenance clear at a glance, I plotted the dataset size across years.

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(x="year", y="rows", data=df_meta)
plt.title("Dataset Size by Year")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "lab3_plot_rows_by_year.png", dpi=300)
plt.show()

**Observation:**
The plot confirms visually what the table showed: the row counts are consistent around 155-158. There are no sudden drops in country coverage that would skew a time-series or snapshot analysis.

## What Comes Next

The most important finding from this lab is that the raw CSV files are consistent in row count but have schema differences that require the explicit normalization mapping. Lab 4 takes the combined dataset and actually cleans and prepares it for modeling. The provenance documentation here is what I'll point to if someone asks 'how do you know your preprocessing is correct?'